# SEC EDGAR: extract and load

This notebook mirrors the **edgar-data-parse** pipeline:

1. **Extract** — `SecEdgarClient` (`src/sec_edgar/client.py`) and shims in `src/fetch.py` / `src/parse.py` (company facts JSON, optional HTM).
2. **Load (optional)** — `sync_company_facts_to_db` into `warehouse.Fact` (same as `POST /api/v1/companies/{id}/sync-facts/`).

**Docs in the repo:** [docs/README.md](../docs/README.md) (index), [docs/api-and-cli.md](../docs/api-and-cli.md) (routes and `manage.py` commands), [docs/architecture.md](../docs/architecture.md) (data flow).

[SEC API documentation](https://www.sec.gov/edgar/sec-api-documentation)

**Prerequisites**

- Activate the project virtualenv; install deps from `requirements.txt` at the repo root (includes Django, `tenacity`, `beautifulsoup4`, `lxml`).
- For the optional Django section: from `src/`, run `python manage.py migrate` once so SQLite (or your `DATABASE_URL` DB) has tables.
- Set `USER_AGENT_EMAIL` in the environment (or in a `.env` loaded below) so requests identify you per [SEC fair access](https://www.sec.gov/os/webmaster-faq#code-support).


## 1. Repo paths and SEC user agent

The notebook adds `src/` to `sys.path` so `fetch`, `parse`, `sec_edgar`, etc. import like `src/main.py`.


In [5]:
from __future__ import annotations

import os
import sys
from pathlib import Path

# Resolve repo root (directory that contains src/manage.py)
_cwd = Path.cwd().resolve()
REPO_ROOT = _cwd
if not (REPO_ROOT / "src" / "manage.py").exists():
    for p in _cwd.parents:
        if (p / "src" / "manage.py").exists():
            REPO_ROOT = p
            break
    else:
        raise RuntimeError("Run the kernel with cwd inside the repo, or open the notebook from edgar-data-parse.")

SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

_dotenv_path = REPO_ROOT / ".env"
try:
    from dotenv import load_dotenv
    _loaded = load_dotenv(_dotenv_path, override=True)
    if not _loaded:
        print(f"⚠ .env file not found at {_dotenv_path}")
except ImportError:
    print("⚠ python-dotenv is not installed — .env will NOT be loaded. Run: pip install python-dotenv")

# SEC client reads USER_AGENT_EMAIL (see sec_edgar.client._user_agent)
if not os.environ.get("USER_AGENT_EMAIL"):
    os.environ["USER_AGENT_EMAIL"] = "you@example.com"  # replace or set in shell / .env

print("REPO_ROOT:", REPO_ROOT)
print("USER_AGENT_EMAIL:", os.environ.get("USER_AGENT_EMAIL", ""))


REPO_ROOT: /Users/bamr87/github/edgar-data-parse
USER_AGENT_EMAIL: amr@bash-365.com


## 2. Extract — ticker, CIK, company facts

Uses `SecEdgarClient` / `fetch.get_facts` (no duplicate `requests` helpers).


In [7]:
TICKER = "TSLA"  # Valmont; change as needed

from sec_edgar.client import SecEdgarClient
from fetch import cik_ticker, get_facts

client = SecEdgarClient()
info = client.cik_for_ticker(TICKER)
cik = info["cik"]
print("CIK:", cik, "|", info["name"])

payload = get_facts(TICKER)
print("entityName:", payload.get("entityName"))
print("top-level keys:", sorted(payload.keys()))
facts_root = payload.get("facts") or {}
print("taxonomies:", sorted(facts_root.keys())[:12], "...")


CIK: 0001318605 | Tesla, Inc.
entityName: Tesla, Inc.
top-level keys: ['cik', 'entityName', 'facts']
taxonomies: ['dei', 'us-gaap'] ...


## 3. Long-format DataFrame — `parse.facts_DF`

All taxonomies and concepts in one table; preview only (no full taxonomy dump).


In [8]:
import pandas as pd
from parse import facts_DF

df, meta = facts_DF(TICKER, None)
print("rows:", len(df), "| columns:", list(df.columns))
print("meta:", meta)
display(df.head(10))

# Example: common revenue tags (see data/reference/financial_model.json)
REVENUE_TAGS = [
    "Revenues",
    "RevenueFromContractWithCustomerExcludingAssessedTax",
    "SalesRevenueNet",
    "RevenueFromContractWithCustomerIncludingAssessedTax",
]
rev = df[(df["taxonomy"] == "us-gaap") & (df["concept"].isin(REVENUE_TAGS))]
display(rev.sort_values("end", ascending=False).head(15))


rows: 23454 | columns: ['taxonomy', 'concept', 'unit', 'end', 'val', 'form']
meta: {'entityName': 'Tesla, Inc.', 'cik': 1318605}


,taxonomy,concept,unit,end,val,form
0,dei,EntityCommonStockSharesOutstanding,shares,2011-07-31,104018274.0,10-Q
1,dei,EntityCommonStockSharesOutstanding,shares,2011-10-31,104298634.0,10-Q
2,dei,EntityCommonStockSharesOutstanding,shares,2012-01-31,104604044.0,10-K
3,dei,EntityCommonStockSharesOutstanding,shares,2012-01-31,104604044.0,10-K/A
4,dei,EntityCommonStockSharesOutstanding,shares,2012-04-30,105214400.0,10-Q
5,dei,EntityCommonStockSharesOutstanding,shares,2012-07-16,105432497.0,10-Q
6,dei,EntityCommonStockSharesOutstanding,shares,2012-10-31,113778865.0,10-Q
7,dei,EntityCommonStockSharesOutstanding,shares,2013-01-31,114517973.0,10-K
8,dei,EntityCommonStockSharesOutstanding,shares,2013-04-30,115552703.0,10-Q
9,dei,EntityCommonStockSharesOutstanding,shares,2013-07-31,121449647.0,10-Q


,taxonomy,concept,unit,end,val,form
19603,us-gaap,Revenues,USD,2025-12-31,9.482700e+10,10-K
19327,us-gaap,RevenueFromContractWithCustomerExcludingAssess...,USD,2025-12-31,9.482700e+10,10-K
19326,us-gaap,RevenueFromContractWithCustomerExcludingAssess...,USD,2025-09-30,2.809500e+10,10-Q
19601,us-gaap,Revenues,USD,2025-09-30,6.992600e+10,10-Q
19602,us-gaap,Revenues,USD,2025-09-30,2.809500e+10,10-Q
19325,us-gaap,RevenueFromContractWithCustomerExcludingAssess...,USD,2025-09-30,6.992600e+10,10-Q
19324,us-gaap,RevenueFromContractWithCustomerExcludingAssess...,USD,2025-06-30,2.249600e+10,10-Q
19599,us-gaap,Revenues,USD,2025-06-30,4.183100e+10,10-Q
19323,us-gaap,RevenueFromContractWithCustomerExcludingAssess...,USD,2025-06-30,4.183100e+10,10-Q
19600,us-gaap,Revenues,USD,2025-06-30,2.249600e+10,10-Q


## 4. (Optional) Submissions index — compact preview

Recent filing metadata from `data.sec.gov/submissions/` — not the full JSON.


In [9]:
sub = client.submissions(cik)
print("submissions keys (sample):", sorted(sub.keys())[:20])

recent = (sub.get("filings") or {}).get("recent") or {}
forms = recent.get("form")
if forms:
    import pandas as pd
    filings_preview = pd.DataFrame(
        {
            "form": list(forms)[:15],
            "filingDate": list((recent.get("filingDate") or [])[:15]),
            "accessionNumber": list((recent.get("accessionNumber") or [])[:15]),
        }
    )
    display(filings_preview)
else:
    print("(No filings block in expected shape)")


submissions keys (sample): ['addresses', 'category', 'cik', 'description', 'ein', 'entityType', 'exchanges', 'filings', 'fiscalYearEnd', 'flags', 'formerNames', 'insiderTransactionForIssuerExists', 'insiderTransactionForOwnerExists', 'investorWebsite', 'lei', 'name', 'ownerOrg', 'phone', 'sic', 'sicDescription']


,form,filingDate,accessionNumber
0,4,2026-03-09,0001104659-26-025379
1,144,2026-03-06,0001950047-26-002335
2,4,2026-02-27,0001104659-26-021746
3,144,2026-02-25,0001950047-26-001763
4,10-K,2026-01-29,0001628280-26-003952
5,8-K,2026-01-28,0001628280-26-003837
6,4,2026-01-12,0001972928-26-000001
7,4,2026-01-06,0001104659-26-001460
8,144,2026-01-02,0001968582-26-000010
9,8-K,2026-01-02,0001628280-26-000016


## 5. (Optional) HTM filing — download and `parse_sec_htm`

Downloads one document to `data/` and parses **Item** sections (see `sec_edgar.parsers.htm`). Set `SEC_HTM_URL` to a direct `.htm` link from EDGAR, or skip if unset.


In [10]:
import os
from fetch import download_filing
from parse import parse_sec_htm

htm_url = os.environ.get("SEC_HTM_URL", "").strip()
if not htm_url:
    print("Set SEC_HTM_URL to a filing document URL to run this cell, e.g. a 10-K primary HTML.")
else:
    data_dir = REPO_ROOT / "data"
    data_dir.mkdir(parents=True, exist_ok=True)
    name = htm_url.rstrip("/").split("/")[-1] or "filing.htm"
    local = data_dir / name
    download_filing(htm_url, str(local))
    parsed = parse_sec_htm(local)
    print("Sections found:", len(parsed.get("sections", {})))
    for title, text in list(parsed.get("sections", {}).items())[:3]:
        preview = (text or "")[:400].replace("\n", " ")
        print("\n---", title[:80], "---\n", preview, "...")


Set SEC_HTM_URL to a filing document URL to run this cell, e.g. a 10-K primary HTML.


## 6. Optional — Django warehouse + sync facts

Requires `python manage.py migrate` (from `src/`). This mirrors `CompanyViewSet` action `sync-facts` in `api/v1/views.py`.


In [ ]:
import os

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")
import django

django.setup()

from warehouse.models import Company
from sec_edgar.services.company_facts import sync_company_facts_to_db

company, created = Company.objects.get_or_create(
    cik=cik,
    defaults={
        "ticker": TICKER.upper(),
        "name": (payload.get("entityName") or info["name"])[:255],
    },
)
if not created and company.ticker != TICKER.upper():
    company.ticker = TICKER.upper()
    company.save(update_fields=["ticker", "updated_at"])

n_facts = sync_company_facts_to_db(company, payload)
print("facts_loaded:", n_facts)


## 7. Alternative — call the REST API

If `runserver` is up, e.g. `POST /api/v1/companies/{id}/sync-facts/` loads facts for a company. Uncomment and set `API_BASE` and `COMPANY_PK`.


In [ ]:
# import requests
# API_BASE = "http://127.0.0.1:8000"
# COMPANY_PK = 1
# r = requests.post(f"{API_BASE}/api/v1/companies/{COMPANY_PK}/sync-facts/")
# r.raise_for_status()
# print(r.json())


---

### Appendix — bulk offline data (do not run by default)

The SEC publishes [companyfacts.zip](https://www.sec.gov/Archives/edgar/daily-index/xbrl/companyfacts.zip) for bulk downloads. Processing the full archive belongs in batch jobs, not this demo notebook (avoid printing or storing entire taxonomies here).
